In [ ]:
!pip install chromadb openai

In [ ]:
from chromadb import PersistentClient
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

chroma_embedding_function = OpenAIEmbeddingFunction(
    api_key=userdata.get('OPENROUTER_API_KEY'),
    model_name="text-embedding-ada-002",
    api_base="https://openrouter.ai/api/v1"
)
chroma_client = PersistentClient("/content/sample_data")
#chroma_collection = chroma_client.get_or_create_collection("my_collection")
chroma_collection = chroma_client.get_or_create_collection("my_collection", embedding_function=chroma_embedding_function)

In [15]:

from pathlib import Path
#FILES = [Path("/content/Methamorphosis.txt"), Path("/content/War and Pease.txt")]
FILES = [Path("/content/Metamorphosis.txt")]


In [22]:
#def index_file(path: Path):
#   with path.open() as file:
#        content = file.read()
#    #print(len(path))

#   chroma_collection.add(
#        ids=[path.stem],
#        documents=[content]
#    )

def index_file(path: Path):
   with path.open() as file:
        lines = file.readlines()
    #print(len(path))

   lines = [line.strip() for line in lines]
   lines = [line for line in lines if line !=""]

   CHUNK_LENGHT = 7
   CHUNK_OVERLAP = 2

   line_idx = 0
   chunks = []
   while line_idx < len(lines):
    chunks.append(" ".join(lines[line_idx:line_idx + CHUNK_LENGHT]))
    line_idx += CHUNK_LENGHT - CHUNK_OVERLAP


   BATCH_SIZE = 800
   for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]
    chroma_collection.add(
      ids=[f"{path.stem}_{i + j + 1}" for j in range(len(BATCH_SIZE))],
      documents=batch
  )

   #chroma_collection.add(
   #    ids=[path.stem],
   #     documents=[content]
   # )




In [25]:
chroma_collection.count()

2

In [ ]:
#for path in FILES:
#   index_file(path)

for path in FILES:
  with path.open() as file:
    line = file.readlines()
    print(file.readlines())



In [8]:
chroma_collection.add(
    ids=["first"],
    documents=["Hello world!"]
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 87.9MiB/s]


In [24]:
chroma_collection.query(query_texts=["greetings", "globalization"])

{'ids': [['first', 'second'], ['first', 'second']],
 'embeddings': None,
 'documents': [['Hello world!', 'It is beautifull day outside!'],
  ['Hello world!', 'It is beautifull day outside!']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None], [None, None]],
 'distances': [[0.9303125739097595, 1.5089715719223022],
  [1.4575889110565186, 1.898533582687378]]}

In [23]:
chroma_collection.add(
    ids=["second"],
    documents=["It is beautifull day outside!"]
)

In [ ]:
from openai import OpenAI
from google.colab import userdata
openrouter_client = OpenAI(
    api_key=userdata.get('OPENROUTER_API_KEY'), base_url="https://openrouter.ai/api/v1")
)

In [ ]:
openrouter_client.model.list()

In [ ]:
from pydantic import BaseModel, Field

class LookupRequest(BaseModel):
  query: str = Field(description="The query to lookup in natural language")



In [ ]:
TOOLS = [
    {
        "type": "function",
        "name": "lookup",
        "description": "A search engine. Useful for when you need to answer questions about current events. Input should be a search query.",
        "parameters": LookupRequest.model_json_schema()
    }
]


def query_library(arguments: str):
  #lookup_request = LookupRequest(**json.loads(arguments))
  lookup_request = LookupRequest.model_validate_json(arguments)
  print(f"Initializing lookup with query: {lookup_request.query}")
  query_result = chroma_collection.query(query_texts=[lookup_request.query], n_results=5)

  matches = query_result['document'][0]
  return "\n\n".join(matches)


  TOOL_EXECUTORS = {
      "lookup": query_library
  }




In [26]:
from typing import List

def invoke(model: str, input: List, max_iteration: int = 20):
  for _ in range(max_iteration):
    current_response = openrouter_client.responses.create(
        model=model,
        input=input,
        tools=TOOLS
  )
    if current_response.output_text != "":
      print(current_response.output_text)
    conversation_history.extend(current_response.output)
    tool_calls = [x for x in current_response.output if x.type == "function_call"]

    if len(tool_calls) == 0:
      return

    while len(tool_calls) > 0:
      for x in tool_calls:
        #if x.name in TOOL_EXECUTORS:
        #  executor = TOOL_EXECUTORS[x.name]
        #  outcome = executor(x.arguments)

        executor = TOOL_EXECUTORS[x.name]
        outcome = executor(x.arguments)

        input.append({
            "type": "function_call_output",
            "output": outcome,
            "call_id": x.call_id
      })

        next_response = openrouter_client.responses.create(
            model=model,
            input=input,
            tools=TOOLS
      )
        if next_response.output_text != "":
          print(next_response.output_text)

        input.extend(next_response.output)
        tool_calls = [x for x in next_response.output if x.type == "function_call"]

    raise Exception("Max iteration reached")

In [ ]:
conversations = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Find all quotes about human kindness within my book library"}
]

In [ ]:
invoke("gtp-luna-6.1", conversations)

In [ ]:
conversations.append[{"role": "user", "content": "Ok I understand, now please find me quotes about existential questions."}]

In [ ]:
invoke("gpt-3.5-turbo-1106", conversations)

In [ ]:
find_quote_response = openrouter_client.responses.create(
    model="gpt-3.5-turbo-1106",
    input="Find all quotes about human kindness within books in my library.",
    tools=TOOLS
)

In [ ]:
find_quote_response.output_text

In [ ]:
tool_calls = [x for x in find_quote_response.output if x.type == "function_call"]

In [ ]:
tool_calls

In [ ]:
import json

for x in tool_calls:
  #parsed_arguments = json.loads(x.arguments)
  #print(LookupRequest(**parsed_arguments))
  #parsed_arguments = LookupRequest(**json.loads(x.arguments))
  #if x.name in TOOL_EXECUTORS:
  #  TOOL_EXECUTORS[x.name](x.arguments)

  executor = TOOL_EXECUTORS[x.name]
  outcome = executor(x.arguments)
  print({
      "type": "function_call_output",
      "output": outcome,
      "call_id": x.call_id
  })

